In [1]:
import os
os.chdir("/home/mrufolo/sysid-neural-manifold/")
import pickle
from pathlib import Path
from functools import partial
from argparse import Namespace
from tqdm import tqdm
import scipy
import sys
import jax
import jax.numpy as jnp
import jax.random as jr
import optax
from flax.training import train_state
from dataset.input.signals import multisine_signal
import dataset.dynamics.boucwen as dyn
from dataset.simulate import simulate_rk4 as simulate
from dataset.simulate import generate_batch
from neuralss import ss_init, ss_apply
from ae import VAE_Encoder, Projector
from lr import create_learning_rate_fn

Jax plugin configuration error: Plugin module jax_plugins.xla_cuda12 does not exist


In [2]:
jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [3]:
# Configuration
cfg = {
    # Misc
    "log_wandb": True,

    # Meta dataset
    "K": 2,  # repetitions from the same system, unused
    "nu": 1,
    "ny": 1,
    "seq_len": 1500,
    "skip_sim": 500,
    "fs": 750.0, # sampling time
    "fh": 150, # highest frequency
    "upsamp": 20, # upsampling for integration
    "input_scale": 50,
    "output_scale": 7e-4,

    # Base learner
    "nx": 3,
    "hidden_f": 16,
    "hidden_g": 16,
    
    # Encoder
    "nh": 128,
    "nz": 20,
    
    # Optimization
    "batch_size": 128,  # systems sampled at each meta optimization step
    "iters": 200_000,
    "lr": 2e-4,
    #"alpha": 1.0, # lr reduction for cosine scheduling
    "clip": 1.0,
    "warmup_iters": 0,
    "skip_loss": 500,  # skipped from the loss computation, to avoid a more advanced handling of the initial condition
    "same_sys": 10
}
MAX_BETA = 0.1

cfg = Namespace(**cfg)

In [4]:
if cfg.log_wandb:
    import wandb
    wandb.init(
        project="sysid-VAE-manifold-meta",
        name=f"run_VAE_bw_beta_{MAX_BETA}",
        #name="run1",
        # track hyperparameters and run metadata
        config=vars(cfg)
    )

KeyboardInterrupt: 

In [ ]:
seed = 12345
key = jr.key(seed)
dec_key, proj_key, enc_key, data_key, train_key = jr.split(key, 5)

In [ ]:
# Meta dataset definition
fs_up = cfg.fs * cfg.upsamp
ts_up = 1.0 / fs_up
N = cfg.seq_len + cfg.skip_sim
N_up = N * cfg.upsamp

input_fn = partial(multisine_signal, seq_len=N_up, fs=fs_up, fh=cfg.fh, scale=cfg.input_scale)
simulate_fn = jax.jit(partial(simulate, f_xu=dyn.f_xu))
generate_batch = partial(
    generate_batch,
    init_fn=dyn.init_fn,  # random initial state
    input_fn=input_fn,  # random input
    params_fn=dyn.params_fn,  # random system parameters
    simulate_fn=simulate_fn,  # simulation function
)


def generate_batches(key, batch_size=cfg.batch_size, K=cfg.K):
    generate_batch_cfg = jax.jit(partial(generate_batch, systems=batch_size, runs=K))
    while True:
        key, subkey = jr.split(key, 2)
        yield generate_batch_cfg(subkey)


def preproc_batch(batch):
    batch_u, batch_x, batch_t, batch_params = batch
    batch_y = batch_x[..., [0]]

    batch_u /= cfg.input_scale
    batch_y /= cfg.output_scale

    if cfg.upsamp > 1:
        batch_u = scipy.signal.decimate(batch_u, q=cfg.upsamp, axis=-2)
        batch_y = scipy.signal.decimate(batch_y, q=cfg.upsamp, axis=-2)

    batch_y1 = batch_y[:, 0, cfg.skip_sim:]
    batch_u1 = batch_u[:, 0, cfg.skip_sim:]

    batch_y2 = batch_y[:, 1, cfg.skip_sim:]
    batch_u2 = batch_u[:, 1, cfg.skip_sim:]

    return batch_y1, batch_u1, batch_y2, batch_u2

In [ ]:
# Initialize data loader
train_dl = generate_batches(data_key)
batch = next(iter(train_dl))
batch_y1, batch_u1, batch_y2, batch_u2 = preproc_batch(batch)
batch_y1.shape, batch_u1.shape,batch_y2.shape, batch_u2.shape,

((128, 1500, 1), (128, 1500, 1), (128, 1500, 1), (128, 1500, 1))

In [ ]:
# Initialize state-space model
params_ss = ss_init(dec_key, nu=cfg.nu, ny=cfg.ny, nx=cfg.nx)
params_ss_flat, unflatten_dec = jax.flatten_util.ravel_pytree(params_ss)
n_params = params_ss_flat.shape[0]
scalers = {"f": {"lin": 1e-2, "nl": 1e-2}, "g": {"lin": 1e0, "nl": 1e0}}

In [ ]:
enc = VAE_Encoder(mlp_layers=[cfg.nh, cfg.nz], rnn_size=cfg.nh)
proj = Projector(outputs=n_params,  unflatten=unflatten_dec)

params_log_sigma = jnp.zeros(()) 
params_enc = enc.init(enc_key, jnp.ones((cfg.seq_len, cfg.ny)), jnp.ones((cfg.seq_len, cfg.nu)))
params_proj = proj.init(enc_key, jnp.ones((cfg.nz,)))
params_proj = jax.tree_util.tree_map(lambda w: w / 20.0, params_proj)

In [ ]:
def instance_loss_fn(p, y1, u1, y2, u2,key,beta):

    p_enc, p_proj, p_ss,log_sigma_est = p
    sigma_noise = jnp.exp(log_sigma_est)

    # Encode (y1, u1) in z and project
    enc_mean, enc_logstd  = enc.apply(p_enc, y1, u1)
    enc_std = jnp.exp(enc_logstd)
    z = enc_mean + jr.normal(key, enc_mean.shape) * enc_std
    p_ss_proj = proj.apply(p_proj, z)

    # Sum the base ss parameters
    p = jax.tree.map(lambda x, y: x+y, p_ss, p_ss_proj)

    # Simulate
    x0 = jnp.zeros((cfg.nx, ))
    y2_hat = ss_apply(p, scalers, x0, u2)

    # Compute loss
    scaled_err = (y2 - y2_hat)/(sigma_noise+1e-6)
    delta = (1.0)/(sigma_noise+1e-6)
    huber_penalty = optax.huber_loss(scaled_err[cfg.skip_loss:], delta=delta)

    # Replace the 0.5 * sum(squared_error) with the sum of the Huber penalty
    nll_loss = 0.5 *(jnp.sum(huber_penalty) 
                + y2.shape[0] * log_sigma_est 
                + (y2.shape[0] / 2) * jnp.log(2 * jnp.pi))
    # nll_loss = 0.5 * jnp.sum((scaled_err[cfg.skip_loss:] )**2) + y2.shape[0] * log_sigma_est + y2.shape[0]/2 * jnp.log(2*jnp.pi)
    kl_loss = 0.5 * jnp.mean(enc_mean**2 + enc_std**2 - 2 * jnp.log(enc_std) - 1) # KL(N(mean, std^2) || N(0, 1))
    scale = 0.5 #* y2.shape[0]
    nll_loss = nll_loss / (scale* y2.shape[0])
    kl_loss = kl_loss / scale
    mse_loss = jnp.mean((y2[cfg.skip_loss:] - y2_hat[cfg.skip_loss:]) ** 2)

    loss = nll_loss + beta * kl_loss
    return loss, (nll_loss, kl_loss, mse_loss)


# batched loss
def loss_fn(*args):
    loss, (nll_loss, kl_loss, mse_loss) = jax.vmap(instance_loss_fn, in_axes=(None, 0, 0, 0, 0, 0, None))(*args)
    return jnp.mean(loss), (jnp.mean(nll_loss), jnp.mean(kl_loss), jnp.mean(mse_loss))

In [ ]:
#lr_scheduler = create_learning_rate_fn(cfg)

opt = optax.chain(
  # optax.clip(cfg.clip),
  optax.clip_by_global_norm(cfg.clip),
  optax.adam(learning_rate=cfg.lr),
)
state = train_state.TrainState.create(apply_fn=loss_fn, params=(params_enc, params_proj, params_ss,params_log_sigma), tx=opt)

@jax.jit
def make_step(state, y1, u1, y2, u2, train_key, beta):
        train_key, train_subkey = jr.split(train_key, 2) # subkey to be consumed at the current iteration
        train_keys = jr.split(train_subkey, cfg.batch_size)  # we need one key per sample in the batch
        (loss, (nll_loss, kl_loss,mse_loss)), grads = jax.value_and_grad(state.apply_fn, has_aux=True)(state.params, y1, u1, y2, u2,train_keys,beta)
        state = state.apply_gradients(grads=grads)
        return loss, nll_loss, kl_loss,mse_loss, state, train_key

In [ ]:
LOSS = []
loss = jnp.array(jnp.nan)
#for itr, batch in (pbar := tqdm(enumerate(train_dl), total=cfg.iters)):
try:
    for itr in (pbar := tqdm(range(cfg.iters))):

        if itr % cfg.same_sys == 0: # some speed up
            batch = next(iter(train_dl))
            batch_y1, batch_u1, batch_y2, batch_u2 = preproc_batch(batch)
        
        current_beta =jnp.minimum(MAX_BETA, jnp.maximum(0.0, MAX_BETA*((itr - 1000) / 21000.0)))

        loss, nll_loss, kl_loss,mse_loss, new_state, train_key = make_step(state, batch_y1, batch_u1, batch_y2, batch_u2, train_key, current_beta)
        if not jnp.isnan(loss).any() and loss < 100.0: # fix some instability issues in training
            state = new_state

        LOSS.append(loss.item())
        if itr % 10 == 0:
            pbar.set_postfix_str(
                f"loss:{loss.item():.4f},mse:{mse_loss.item():.4f}"
            )
            
        #if itr % 100 == 0 and cfg.log_wandb:
        if cfg.log_wandb:
            if itr % 1 == 0:
                wandb.log({
                        'mse': float(mse_loss),
                        'VAE_loss': float(loss),
                        'nll': float(nll_loss),
                        'kl': float(kl_loss),
                        'beta': float(current_beta) # Useful to track your annealing!
                    })

        if itr % 5000 == 0:
            ckpt = {
                "cfg": cfg,
                "params": state.params,
                "scalers": scalers,
                "LOSS": jnp.array(LOSS),
            }
            ckpt_path = Path("tmp") / f"vae_{itr}.p"
            ckpt_path.parent.mkdir(exist_ok=True, parents=True)
            pickle.dump(ckpt, open(ckpt_path, "wb"))


        if itr == cfg.iters:
            break
except KeyboardInterrupt:
    print("\n[!] Ctrl+C detected. Closing gracefully...")
    
finally:
    # --- THIS BLOCK ALWAYS RUNS ---
    # Whether it finishes normally OR you press Ctrl+C, this block is guaranteed to execute
    print("Saving final checkpoint...")
    
    ckpt = {
        "cfg": cfg,
        "params": state.params,
        "scalers": scalers,
        "LOSS": jnp.array(LOSS),
    }
    ckpt_path = Path("out") / f"vae_{MAX_BETA}.p"
    ckpt_path.parent.mkdir(exist_ok=True, parents=True)
    pickle.dump(ckpt, open(ckpt_path, "wb"))
    print(f"Checkpoint saved to {ckpt_path}")

    if cfg.log_wandb:
        print("Syncing WandB...")
        wandb.finish()
        
    print("Exiting.")
    # sys.exit()

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [01:46<00:00,  5.32s/it, loss:2.3591,mse:0.7842]
